# LoRA / PEFT Fine-Tuning — AI Engineering Interview Prep

Low-Rank Adaptation concepts, implementation, and parameter efficiency analysis.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
print(f"PyTorch {torch.__version__}")

## 1. The Problem: Full Fine-Tuning is Expensive

In [ ]:
# Model parameter counts for reference
param_counts = {
    'GPT-2 (small)':   124e6,
    'GPT-2 (large)':   774e6,
    'BERT-base':       110e6,
    'LLaMA 2 7B':      7e9,
    'LLaMA 2 70B':     70e9,
    'GPT-4 (est.)':    1.8e12,
}

print("Model sizes and full fine-tuning memory (fp32):")
print(f"{'Model':<20} {'Params':>12} {'Memory (fp32)':>15} {'Memory (fp16)':>15}")
print("-" * 65)
for name, params in param_counts.items():
    mem_fp32 = params * 4 / 1e9   # 4 bytes per float32
    mem_fp16 = params * 2 / 1e9   # 2 bytes per float16
    print(f"{name:<20} {params/1e9:>10.1f}B {mem_fp32:>12.1f}GB {mem_fp16:>12.1f}GB")

## 2. LoRA: Low-Rank Adaptation

In [ ]:
"""
LoRA key insight:
  W_pretrained is large (d x k), frozen
  ΔW = B @ A  where B is (d x r) and A is (r x k), r << min(d,k)
  W_new = W_pretrained + (alpha/r) * B @ A

  Parameters: d*k (original) → r*d + r*k (LoRA) = r*(d+k)
"""

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=4, alpha=1.0):
        super().__init__()
        self.rank  = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        # Frozen pretrained weights
        self.weight = nn.Parameter(
            torch.randn(out_features, in_features), requires_grad=False
        )

        # Trainable LoRA matrices
        self.lora_A = nn.Parameter(torch.randn(rank, in_features) * 0.02)
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))

    def forward(self, x):
        # Original weights (frozen) + LoRA adaptation
        return (x @ self.weight.T +
                x @ self.lora_A.T @ self.lora_B.T * self.scaling)

    def trainable_params(self):
        return self.lora_A.numel() + self.lora_B.numel()

    def total_params(self):
        return self.weight.numel() + self.trainable_params()

# Example: attention projection layer
d_model, rank = 768, 8  # BERT-base hidden size
lora_layer = LoRALinear(d_model, d_model, rank=rank, alpha=16)

orig_params = d_model * d_model
lora_params = lora_layer.trainable_params()

print(f"d_model={d_model}, rank={rank}")
print(f"Original params:  {orig_params:>10,}")
print(f"LoRA params:      {lora_params:>10,}")
print(f"Reduction:        {lora_params/orig_params:.4f}x ({(1-lora_params/orig_params)*100:.1f}% reduction)")

x = torch.randn(4, 32, d_model)
out = lora_layer(x)
print(f"\nForward pass: {x.shape} → {out.shape}")

## 3. Parameter Efficiency Comparison

In [ ]:
import pandas as pd

# For a model with n attention layers and d_model
def lora_param_analysis(n_layers=12, d_model=768, ranks=[1,2,4,8,16,32,64]):
    # Each layer has Q,K,V,O projection matrices
    full_params_per_layer = 4 * d_model * d_model
    full_params_total = n_layers * full_params_per_layer

    results = []
    for r in ranks:
        lora_per_layer = 4 * 2 * r * d_model  # 4 matrices, 2 LoRA matrices each
        lora_total = n_layers * lora_per_layer
        results.append({
            'rank': r,
            'trainable_params': lora_total,
            'total_model_params': full_params_total,
            'pct_trainable': lora_total / full_params_total * 100,
            'memory_MB_fp16': lora_total * 2 / 1e6
        })
    return pd.DataFrame(results)

df = lora_param_analysis(n_layers=12, d_model=768)
print("LoRA parameter analysis (BERT-base, 12 layers, d=768):")
print(df.to_string(index=False, float_format='{:.3f}'.format))

plt.figure(figsize=(8, 4))
plt.plot(df['rank'], df['pct_trainable'], 'bo-')
plt.xlabel('LoRA Rank'); plt.ylabel('% Trainable Parameters')
plt.title('LoRA Parameter Efficiency vs Rank')
plt.grid(True)
plt.show()

## 4. PEFT with Hugging Face (Conceptual Code)

In [ ]:
# This is how you'd use PEFT in practice (requires peft and transformers)
peft_example_code = '''
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

# Load base model
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf")

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                          # rank
    lora_alpha=32,                # scaling = alpha/r
    target_modules=["q_proj", "v_proj"],  # which layers to adapt
    lora_dropout=0.05,
    bias="none",
    inference_mode=False,
)

# Wrap model with LoRA
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()
# Output: trainable params: 4,194,304 || all params: 6,742,609,920 || trainable%: 0.0622

# Training is standard — only LoRA params update
# After training:
peft_model.save_pretrained("./lora-llama2-7b")

# Merge LoRA weights back into model for inference
merged_model = peft_model.merge_and_unload()
'''

print("PEFT usage example (requires: pip install peft transformers):")
print(peft_example_code)

## Interview Tips

- **LoRA intuition**: weight updates have low intrinsic rank; ΔW=BA captures the important update directions.
- **Rank selection**: r=4-16 works well for most tasks. Higher r = more expressive but more params.
- **alpha/r scaling**: keeps effective learning rate stable as rank changes. Typical: alpha=2*r.
- **Target modules**: Q and V projections are most impactful; K and O add marginal gains.
- **QLoRA**: combines 4-bit quantization + LoRA, enabling 70B fine-tuning on single 48GB GPU.
- **When NOT to use LoRA**: domain shift is very large, or you have ample compute for full fine-tuning.